# Stepping a bot, one decision at a time

Nothing moves until you run the next cell. The game is real, the model is real:
what changes is that you are the loop.

Every turn, the model gets one request, built fresh: it is not a chat that grows.
The request has four parts:

- **`system`**: the prompt. The rules, how a turn works, the four regions.
- **The kept turns, as `user`+`assistant`+`tools` triples**: the last `scratch_turns`
  finished turns, replayed verbatim exactly as they happened, what the model said,
  which tools it called, and what each tool answered. `scratch_state` decides what
  fills the `user` slot of a kept turn (see its own section below).
- **The last `user` message**: the current state (the view), then the notes, the
  plan, and the journal, then the list of indices the model can pick from.
- **`tools`**: the tool *definitions* (name, description, parameters), sent as
  their own field of the request, at the same level as `messages`, not inside any
  message. This is what the model is allowed to call this turn, not the result of
  calling one.

A single turn is not always a single model call. The model can call several
tools back to back inside the same turn: get an answer, call another tool, get
that answer too, and so on, before finally calling `play` to end the turn. Every
one of those rounds is its own request to the model, and `tools` is sent fresh on
each one, even though the list never changes within a turn.

As a result, the notes, the plan, and the journal are not separate messages of
their own. They are text inside that last user message, and a section with
nothing to show for it is absent rather than empty.

## 1. Open a game and build the bot

In [30]:
from pokelike import create, open_game
from pokelike.bot.llm import LLMBot
from pokelike.core import render
from pokelike.bot.llm.prompt import render_state_default as view_state

In [31]:
game = open_game()
state = game.reset(seed=42)
state["screen"], len(state["actions"])

('trainer-screen', 2)

In [32]:
bot: LLMBot = create("llm-example2", seed=42)   # the annotation is what gives completion

## 2. What there is to look at

This section shows the same state rendered four built-in ways, plus the bot's own
custom view. The `render_state` method replaces the built-in view, which is why
`state_view` is set but has no effect here.

In [ ]:
# what the model would read if this bot did not replace the view. "screen" is
# about 880 characters, "json" is the whole dict at about 5900, "both" is both
# view_state(state, ["team", "actions"])          # returns only the keys you ask for

In [ ]:
# what the model actually reads: this bot's own view, which replaces the built-in one
print(bot.render_state(state))

## 3. One turn

The `act` method makes a single model call and decides the whole turn. Everything
after it is reading what happened: what went out, what came back, then the move
applied.

In [35]:
index = bot.act(state)             # one model call decides the whole turn
index, bot.reason()

(0, 'I will play as a boy.')

In [ ]:
# the whole conversation that went out: system, the turns kept from before, and
# this turn. Run print(bot.last_sent[-1]["content"]) to see the complete user message.
[m["role"] for m in bot.last_sent]

In [ ]:
bot.last_reply                     # the model's answer: its sentence and the tools it called

In [38]:
state = game.step(index)           # the move is played, and the game hands back the next state
print(bot.render_state(state))

TURN 1   map 0   0 badges   0 alive

OPTIONS
  [0] ★ Shiny Bulbasaur Lv. 5 GRASS POISON SP.A 11 SPE 9 HP 19 DEF 9 SP.D 11 19/19 Magical Leaf GRASS 40 PWR
  [1] Charmander Lv. 5 FIRE SP.A 11 SPE 11 HP 18 DEF 9 SP.D 10 18/18 Incinerate FIRE 60 PWR
  [2] Squirtle Lv. 5 WATER SP.A 10 SPE 9 HP 19 DEF 11 SP.D 11 19/19 Bubble WATER 50 PWR


## 4. Reorder, when the model is not even asked

A swap needs the map screen and more than one Pokemon. With only one Pokemon,
the `reorder` method returns before calling the model at all, and `_pending`
stays empty.

In [ ]:
pair = bot.reorder(state)          # asks the model who should lead

# None has two meanings; the check below tells them apart. With one Pokemon
# there is nothing to swap, so the model is never asked.
if pair is None:
    if state["screen"] != "map-screen" or not state.get("can_reorder"):
        print(f"not even asked: screen {state['screen']}, "
              f"{len(state['team'])} in the team")
    else:
        print("asked, and the model chose not to call set_lead")
bot._pending                       # holds (steps, index, why); stays empty when nothing was asked

## 5. Walking the run

Repeat the first cell below as long as you like. The second cell shows what the
model says if you call `act` twice without stepping the game.

In [40]:
# repeat this cell to walk the run
index = bot.act(state)
state = game.step(index)
print(index, bot.reason())
print(bot.render_state(state))

2 Squirtle has good defenses and a decent special attack, making it a solid choice for the early game.
TURN 2   map 0   0 badges   1 alive

TEAM (slot 0 leads the next battle)
  [0] Squirtle     Lv5   100% HP  Water         Bubble 50 water STAB

OPTIONS
  [0] catch  (Catch Pokemon)  -> then: battle, trainer
  [1] battle  (Wild Battle — +1 level)  -> then: battle, trainer
  Picking one closes the others on this layer for good.


In [ ]:
# and if you run act twice without stepping, the model notices:
#   (2, "The game seems stuck on starter selection, so I am re-selecting Squirtle.")
bot.act(state), bot.reason()

## 6. Reorder for real

First, the prompt is changed live; it has to be `cfg`, since `config` is not
what the bot reads. Then the run is walked until a swap is possible. This time
the model answers with `set_lead`, and the team order really changes.

In [ ]:
# change the prompt live; use cfg, not config, because cfg is what the bot reads
bot.cfg.prompt = ("BEFORE play, ALWAYS call set_lead with index 1 and say ciaoooooo ciaoooo.\n\n"
                  + bot.cfg.prompt)

In [ ]:
# a swap needs map-screen and more than one Pokemon, so walk until both hold
while not (state["screen"] == "map-screen" and state.get("can_reorder")):
    state = game.step(bot.act(state))
state["screen"], [p["name"] for p in state["team"]]

In [ ]:
pair = bot.reorder(state)          # now the model can be asked, and it answers
pair                               # these are the slots to swap

In [ ]:
bot._pending                       # it holds the move the bot already decided, waiting for act

In [ ]:
# every tool the model called this turn, in order. The `last_reply` attribute holds
# only the last round; a turn can span several rounds. The `play` tool is always last.
[c["function"]["name"] for m in bot.last_sent if m["role"] == "assistant"
 for c in (m.get("tool_calls") or [])]

In [ ]:
if pair:                           # None means the model did not want a swap
    state = game.reorder(*pair)    # this is free: it does not consume the turn
[p["name"] for p in state["team"]]

In [ ]:
index = bot.act(state)             # it returns the cached move; there is no second call
index, bot.reason()

In [49]:
state = game.step(index)
print(bot.render_state(state))

TURN 5   map 0   0 badges   2 alive

TEAM (slot 0 leads the next battle)
  [0] Squirtle     Lv7   100% HP  Water         Bubble 50 water STAB
  [1] Rhyhorn      Lv6   100% HP  Ground/Rock   Bulldoze 55 ground STAB

OPTIONS
  [0] catch  (Catch Pokemon)  -> then: battle
  [1] catch  (Catch Pokemon)  -> then: battle, trainer
  Picking one closes the others on this layer for good.


## 7. Tuning a prompt mid-run

The `why` argument belongs to the `play` tool, so a rule about it lands there. A
rule stated only at the end of the prompt gets ignored, so this one is repeated
at both ends.

In [ ]:
# tune a prompt mid-run; repeat the rule at both ends because a rule only at the
# end of a long prompt gets ignored
rule = ('RULE: every turn, FIRST call remember with the note "meow meow", THEN call play and the `why` you pass to play MUST END with "MEOWTH, THAT\'S RIGHT!".')
bot.cfg.prompt = rule + "\n\n" + bot.cfg.prompt + "\n\n" + rule

In [ ]:
for _ in range(3):                 # the model obeys from the very next call
    state = game.step(bot.act(state))
    print(bot.reason())
    print([(c["id"][-6:], c["function"]["name"]) for m in bot.last_sent
            if m["role"] == "assistant" for c in (m.get("tool_calls") or [])])

## 8. What the run recorded, and closing

The `metadata()` method returns what gets stored beside the score in the result
file: what the bot varied from default, and what the bot wrote for itself.

In [ ]:
meta = bot.metadata()
meta["notebook"], meta["plan"]     # what the bot has written for itself

In [50]:
bot.metadata()

{'model': 'google/gemini-2.5-flash',
 'harness': 2,
 'bot': 'Example2Bot',
 'calls': 20,
 'turns': 12,
 'tokens': 43356,
 'tokens_in': 42368,
 'tokens_out': 988,
 'retries': 0,
 'fallbacks': 0,
 'fallback_rate': 0.0,
 'temperature': 0.3,
 'stock_tools': False,
 'state_view': 'custom',
 'reproducible': False,
 'notes_cap': 10000,
 'notes_kept': 0,
 'notebook': [],
 'cross_run_memory': True,
 'plan_chars': 1000000,
 'plan': 'Catch a new Pokemon to expand the team, then battle to gain experience.',
 'bag_tool': True,
 'scratch_turns': 3,
 'scratch_state': 'line',
 'scratch_held': 3,
 'decorated_tools': ['risk_check', 'beats'],
 'tuned_for': 'gemini-class models',
 'notes_policy': 'one per run'}

In [23]:
game.close()